# Lab 10: Client APIs for DuckDB - SOLUTION

This notebook contains the solution for Lab 10 exercises.

## 💻 Exercise 1: Python API Mastery - SOLUTION

### Solution

In [ ]:
import duckdb
import pandas as pd
import time

# Solution for advanced Python API usage

print("Python API Mastery Solution:")
print("="*50)

# 1. Connection pooling simulation
print("\n1. Connection Pooling:")

class ConnectionPool:
    """Simple connection pool for DuckDB"""
    def __init__(self, db_path, pool_size=3):
        self.db_path = db_path
        self.pool = []
        self.pool_size = pool_size
        
        for _ in range(pool_size):
            self.pool.append(duckdb.connect(db_path))
        print(f"✓ Created connection pool with {pool_size} connections")
    
    def get_connection(self):
        if self.pool:
            return self.pool.pop()
        return duckdb.connect(self.db_path)
    
    def return_connection(self, conn):
        if len(self.pool) < self.pool_size:
            self.pool.append(conn)
        else:
            conn.close()

pool = ConnectionPool('data/duckdb_practice.db')

# Use connection from pool
conn1 = pool.get_connection()
result = conn1.execute("SELECT COUNT(*) FROM sample_customers").fetchone()
print(f"✓ Query using pooled connection: {result[0]} customers")
pool.return_connection(conn1)

# 2. Transaction management
print("\n2. Transaction Management:")

con = duckdb.connect('data/transaction_test.db')
con.execute("CREATE TABLE test_trans (id INTEGER, value INTEGER)")

try:
    # Begin transaction
    con.execute("BEGIN TRANSACTION")
    
    # Perform operations
    con.execute("INSERT INTO test_trans VALUES (1, 100)")
    con.execute("INSERT INTO test_trans VALUES (2, 200)")
    
    # Commit transaction
    con.execute("COMMIT")
    print("✓ Transaction committed successfully")
    
except Exception as e:
    con.execute("ROLLBACK")
    print(f"✗ Transaction rolled back: {e}")

count = con.execute("SELECT COUNT(*) FROM test_trans").fetchone()[0]
print(f"✓ Transaction result: {count} rows")

# 3. Error handling
print("\n3. Advanced Error Handling:")

def safe_query(con, query):
    """Execute query with error handling"""
    try:
        result = con.execute(query).df()
        return result
    except duckdb.Error as e:
        print(f"DuckDB Error: {e}")
        return None
    except Exception as e:
        print(f"General Error: {e}")
        return None

# Test error handling
result = safe_query(con, "SELECT * FROM test_trans")
print(f"✓ Safe query executed: {len(result)} rows")

result = safe_query(con, "SELECT * FROM non_existent_table")
print(f"✓ Error handling works: {result}")

# 4. Performance optimization
print("\n4. Performance Optimization:")

con_perf = duckdb.connect(':memory:', config={
    'memory_limit': '2GB',
    'threads': 4,
    'enable_progress_bar': True
})

con_perf.execute("CREATE TABLE large_data AS SELECT * FROM range(1000000)")

# Test query with optimization
start = time.time()
result = con_perf.execute("SELECT SUM(column0) FROM large_data").fetchone()
optimized_time = time.time() - start

print(f"✓ Optimized query: {optimized_time:.4f}s")
print(f"✓ Result: {result[0]}")

# Cleanup
con.close()
con_perf.close()

print("\n✓ Python API mastery demonstrated")

## 💻 Exercise 2: Concurrency Patterns - SOLUTION

### Solution

In [ ]:
import duckdb
import threading
import time

# Solution for concurrency patterns

print("Concurrency Patterns Solution:")
print("="*50)

# 1. Multi-threaded reads
print("\n1. Multi-threaded Reads:")

con = duckdb.connect('data/concurrency_solution.db')
con.execute("CREATE TABLE test_data AS SELECT * FROM range(10000)")

results = []
errors = []

def read_operation(thread_id, con_path):
    """Perform read operation"""
    try:
        local_con = duckdb.connect(con_path, read_only=True)
        result = local_con.execute("SELECT COUNT(*) FROM test_data").fetchone()[0]
        results.append((thread_id, result))
        local_con.close()
    except Exception as e:
        errors.append((thread_id, str(e)))

# Create multiple reader threads
threads = []
for i in range(5):
    thread = threading.Thread(target=read_operation, args=(i, 'data/concurrency_solution.db'))
    threads.append(thread)
    thread.start()

# Wait for all threads to complete
for thread in threads:
    thread.join()

print(f"✓ Completed {len(results)} read operations successfully")
if errors:
    print(f"✗ {len(errors)} errors occurred")

# 2. Batched writes
print("\n2. Batched Writes:")

con.execute("CREATE TABLE batch_test (id INTEGER, value INTEGER)")

def batched_insert(con, batch_size=1000, total_records=5000):
    """Perform batched insert operations"""
    start = time.time()
    
    for i in range(0, total_records, batch_size):
        # Create batch data
        batch_values = [(j, j * 10) for j in range(i, min(i + batch_size, total_records))]
        
        # Insert batch
        for value in batch_values:
            con.execute(f"INSERT INTO batch_test VALUES {value}")
    
    elapsed = time.time() - start
    return elapsed

batch_time = batched_insert(con)
count = con.execute("SELECT COUNT(*) FROM batch_test").fetchone()[0]

print(f"✓ Batched insert: {count} rows in {batch_time:.4f}s")
print(f"✓ Performance: {count/batch_time:.2f} rows/second")

# 3. Connection pooling for concurrency
print("\n3. Connection Pooling:")

class ConcurrentConnectionPool:
    """Thread-safe connection pool"""
    def __init__(self, db_path, pool_size=3):
        self.db_path = db_path
        self.pool = []
        self.pool_size = pool_size
        self.lock = threading.Lock()
        
        for _ in range(pool_size):
            self.pool.append(duckdb.connect(db_path))
    
    def get_connection(self):
        with self.lock:
            if self.pool:
                return self.pool.pop()
            return duckdb.connect(self.db_path)
    
    def return_connection(self, conn):
        with self.lock:
            if len(self.pool) < self.pool_size:
                self.pool.append(conn)
            else:
                conn.close()

pool = ConcurrentConnectionPool('data/concurrency_solution.db')
print("✓ Thread-safe connection pool created")

# Test concurrent access with pool
def pooled_query(thread_id, pool):
    """Query using connection pool"""
    conn = pool.get_connection()
    result = conn.execute("SELECT COUNT(*) FROM test_data").fetchone()[0]
    pool.return_connection(conn)
    return (thread_id, result)

# Test with threads
pooled_results = []
pooled_threads = []

for i in range(5):
    thread = threading.Thread(target=lambda tid, p=p, r=pooled_results: r.append(pooled_query(tid, p)), args=(i,))
    pooled_threads.append(thread)
    thread.start()

for thread in pooled_threads:
    thread.join()

print(f"✓ Pooled queries: {len(pooled_results)} completed")

# 4. Conflict resolution
print("\n4. Conflict Resolution:")

# WAL mode for better concurrency
con_wal = duckdb.connect('data/wal_concurrent.db', config={
    'wal_autocheckpoint': '1GB'
})

con_wal.execute("CREATE TABLE concurrent_test (id INTEGER, value INTEGER)")
print("✓ WAL mode configured for better concurrency")

con_wal.close()
con.close()

print("\n✓ Concurrency patterns demonstrated")